In [5]:


"""
Vanilla GAN Baseline for IDS Minority Class Augmentation
===========================================================
Purpose: De-risk GAN training stability BEFORE building WCGAN/ACGAN.
Trains a separate vanilla GAN per attack class: DoS, Probe, R2L, U2R.

This is Step 1 of your critical path. Goal here is NOT high-quality
synthetic data yet -- goal is confirming training is stable (no mode
collapse, discriminator/generator losses converge reasonably) before
you invest time in the more complex WCGAN/ACGAN architectures.

Author: Vibha J | GAN Implementation Lead
===========================================================
"""

import os
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
import matplotlib.pyplot as plt

# ============================================================
# CONFIG -- EDIT THESE PATHS TO MATCH YOUR LOCAL SETUP
# ============================================================
DATA_DIR = "./data"          # folder containing gan_input_*.csv files
OUTPUT_DIR = "./gan_baseline_outputs"
LATENT_DIM = 32
EPOCHS = 500                  # increase later once stability confirmed
LR = 0.0002
SEED = 42

# Per-class config: (filename, batch_size, epochs)
# Small classes (U2R) need small batch sizes since dataset itself is tiny.
CLASS_CONFIG = {
    "U2R":    {"file": "gan_input_U2R.csv",    "batch_size": 16, "epochs": 800},
    "R2L":    {"file": "gan_input_R2L.csv",    "batch_size": 64, "epochs": 500},
    "Probe":  {"file": "gan_input_Probe.csv",  "batch_size": 128, "epochs": 300},
    "DoS":    {"file": "gan_input_DoS.csv",    "batch_size": 256, "epochs": 200},
}

torch.manual_seed(SEED)
np.random.seed(SEED)
os.makedirs(OUTPUT_DIR, exist_ok=True)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")


# ============================================================
# MODELS
# ============================================================
class Generator(nn.Module):
    def __init__(self, latent_dim, output_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(latent_dim, 64),
            nn.BatchNorm1d(64),
            nn.ReLU(True),
            nn.Linear(64, 128),
            nn.BatchNorm1d(128),
            nn.ReLU(True),
            nn.Linear(128, output_dim),
            nn.Tanh(),
            # Tanh output because scaler.pkl scales numeric features to
            # [-1, 1]. This bounds generator output to match real data
            # range exactly -- standard practice, also helps stability.
        )

    def forward(self, z):
        return self.net(z)


class Discriminator(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 128),
            nn.LeakyReLU(0.2, True),
            nn.Dropout(0.3),
            nn.Linear(128, 64),
            nn.LeakyReLU(0.2, True),
            nn.Dropout(0.3),
            nn.Linear(64, 1),
            nn.Sigmoid(),
        )

    def forward(self, x):
        return self.net(x)


# ============================================================
# TRAINING LOOP FOR ONE CLASS
# ============================================================
def train_gan_for_class(class_name, filepath, batch_size, epochs):
    print(f"\n{'='*60}\nTraining vanilla GAN for class: {class_name}\n{'='*60}")

    df = pd.read_csv(filepath)
    data = df.values.astype(np.float32)
    n_samples, n_features = data.shape
    print(f"Loaded {n_samples} samples, {n_features} features")

    if n_samples < batch_size:
        print(f"WARNING: batch_size ({batch_size}) > n_samples ({n_samples}). "
              f"Reducing batch_size to {n_samples}.")
        batch_size = n_samples

    tensor_data = torch.tensor(data)
    dataset = TensorDataset(tensor_data)
    loader = DataLoader(dataset, batch_size=batch_size, shuffle=True, drop_last=False)

    G = Generator(LATENT_DIM, n_features).to(device)
    D = Discriminator(n_features).to(device)

    opt_G = torch.optim.Adam(G.parameters(), lr=LR, betas=(0.5, 0.999))
    opt_D = torch.optim.Adam(D.parameters(), lr=LR, betas=(0.5, 0.999))
    criterion = nn.BCELoss()

    g_losses, d_losses = [], []

    for epoch in range(epochs):
        epoch_g_loss, epoch_d_loss = 0.0, 0.0
        n_batches = 0

        for (real_batch,) in loader:
            real_batch = real_batch.to(device)
            bs = real_batch.size(0)
            real_labels = torch.ones(bs, 1, device=device)
            fake_labels = torch.zeros(bs, 1, device=device)

            # ---- Train Discriminator ----
            opt_D.zero_grad()
            d_real = D(real_batch)
            loss_d_real = criterion(d_real, real_labels)

            z = torch.randn(bs, LATENT_DIM, device=device)
            fake_batch = G(z).detach()
            d_fake = D(fake_batch)
            loss_d_fake = criterion(d_fake, fake_labels)

            loss_d = loss_d_real + loss_d_fake
            loss_d.backward()
            opt_D.step()

            # ---- Train Generator ----
            opt_G.zero_grad()
            z = torch.randn(bs, LATENT_DIM, device=device)
            fake_batch = G(z)
            d_fake_for_g = D(fake_batch)
            loss_g = criterion(d_fake_for_g, real_labels)  # wants D to think it's real
            loss_g.backward()
            opt_G.step()

            epoch_g_loss += loss_g.item()
            epoch_d_loss += loss_d.item()
            n_batches += 1

        g_losses.append(epoch_g_loss / n_batches)
        d_losses.append(epoch_d_loss / n_batches)

        if (epoch + 1) % max(1, epochs // 10) == 0:
            print(f"Epoch [{epoch+1}/{epochs}]  D_loss: {d_losses[-1]:.4f}  "
                  f"G_loss: {g_losses[-1]:.4f}")

    # ---- Save loss curve (check for mode collapse / instability) ----
    plt.figure(figsize=(8, 4))
    plt.plot(g_losses, label="Generator loss")
    plt.plot(d_losses, label="Discriminator loss")
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.title(f"Vanilla GAN Training Loss - {class_name}")
    plt.legend()
    plt.tight_layout()
    plot_path = os.path.join(OUTPUT_DIR, f"loss_curve_{class_name}.png")
    plt.savefig(plot_path)
    plt.close()
    print(f"Saved loss curve -> {plot_path}")

    # ---- Generate synthetic samples (same count as original, adjust as needed) ----
    G.eval()
    with torch.no_grad():
        z = torch.randn(n_samples, LATENT_DIM, device=device)
        synthetic = G(z).cpu().numpy()

    synthetic_df = pd.DataFrame(synthetic, columns=df.columns)
    synth_path = os.path.join(OUTPUT_DIR, f"synthetic_{class_name}.csv")
    synthetic_df.to_csv(synth_path, index=False)
    print(f"Saved synthetic samples -> {synth_path}")

    # ---- Quick statistical sanity check: real vs synthetic feature means ----
    real_mean = df.mean()
    synth_mean = synthetic_df.mean()
    mean_diff = (real_mean - synth_mean).abs().mean()
    print(f"Mean absolute difference (real vs synthetic feature means): {mean_diff:.4f}")
    print("(Lower is better. Large values may indicate training instability "
          "or insufficient epochs -- inspect the loss curve.)")

    # ---- Save trained generator ----
    model_path = os.path.join(OUTPUT_DIR, f"generator_{class_name}.pt")
    torch.save(G.state_dict(), model_path)
    print(f"Saved generator weights -> {model_path}")

    return {
        "class_name": class_name,
        "n_samples": n_samples,
        "n_features": n_features,
        "final_g_loss": g_losses[-1],
        "final_d_loss": d_losses[-1],
        "mean_abs_diff": mean_diff,
    }


# ============================================================
# MAIN
# ============================================================
if __name__ == "__main__":
    results = []
    for class_name, cfg in CLASS_CONFIG.items():
        filepath = os.path.join(DATA_DIR, cfg["file"])
        if not os.path.exists(filepath):
            print(f"SKIPPING {class_name}: file not found at {filepath}")
            continue
        result = train_gan_for_class(
            class_name=class_name,
            filepath=filepath,
            batch_size=cfg["batch_size"],
            epochs=cfg["epochs"],
        )
        results.append(result)

    # ---- Summary table ----
    print(f"\n{'='*60}\nSUMMARY -- Vanilla GAN Baseline Results\n{'='*60}")
    summary_df = pd.DataFrame(results)
    print(summary_df.to_string(index=False))
    summary_path = os.path.join(OUTPUT_DIR, "baseline_summary.csv")
    summary_df.to_csv(summary_path, index=False)
    print(f"\nSaved summary -> {summary_path}")
    print("\nNEXT STEP: Inspect loss_curve_*.png for each class.")
    print("- D_loss collapsing to ~0 while G_loss explodes = discriminator too strong")
    print("- G_loss collapsing to ~0 = generator fooling D trivially / mode collapse risk")
    print("- Both oscillating but not diverging = healthy vanilla GAN training")
    print("Once stable, move to WCGAN (Wasserstein loss + gradient penalty) for better")
    print("sample diversity, especially on U2R where n_samples is very small (~198).")

Using device: cpu

Training vanilla GAN for class: U2R
Loaded 198 samples, 101 features
Epoch [80/800]  D_loss: 1.3078  G_loss: 0.7883
Epoch [160/800]  D_loss: 1.3081  G_loss: 0.7402
Epoch [240/800]  D_loss: 1.3486  G_loss: 0.7708
Epoch [320/800]  D_loss: 1.3455  G_loss: 0.7634
Epoch [400/800]  D_loss: 1.2954  G_loss: 0.7238
Epoch [480/800]  D_loss: 1.2968  G_loss: 0.7652
Epoch [560/800]  D_loss: 1.3291  G_loss: 0.7593
Epoch [640/800]  D_loss: 1.3309  G_loss: 0.7958
Epoch [720/800]  D_loss: 1.3002  G_loss: 0.7649
Epoch [800/800]  D_loss: 1.3335  G_loss: 0.8163
Saved loss curve -> ./gan_baseline_outputs\loss_curve_U2R.png
Saved synthetic samples -> ./gan_baseline_outputs\synthetic_U2R.csv
Mean absolute difference (real vs synthetic feature means): 0.0281
(Lower is better. Large values may indicate training instability or insufficient epochs -- inspect the loss curve.)
Saved generator weights -> ./gan_baseline_outputs\generator_U2R.pt

Training vanilla GAN for class: R2L
Loaded 2985 samp

In [4]:
dir data

SyntaxError: invalid syntax (187302922.py, line 1)